# Lending Risk Pipeline
Run the cells from top to bottom. This notebook calls reusable functions in `src/transformations.py`.

**Input:** `data/input/lending_risk.csv`  
**Output:** `data/output/`

##  Project Configuration & Imports


In [1]:
from pathlib import Path
import sys,warnings 

from pyspark.sql import SparkSession


PROJECT = Path.cwd()

if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT))

print(f"Project folder: {PROJECT}")
print(f"Source folder exists: {(PROJECT / 'src').exists()}")

from src.transformations import (
    transform_data,
    make_summary,
)

warnings.simplefilter(
    action="ignore",
    category=(FutureWarning,UserWarning)
)

Project folder: C:\lending_risk_overview
Source folder exists: True


##  Start Spark and set paths


In [2]:

INPUT_FILE = PROJECT / "data" / "input" / "lending_risk.csv"

OUTPUT_DIR = PROJECT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Input exists: {INPUT_FILE.exists()}")
print(f"Output folder: {OUTPUT_DIR}")


Input file: C:\lending_risk_overview\data\input\lending_risk.csv
Input exists: True
Output folder: C:\lending_risk_overview\data\output


###  Create Spark Session


In [3]:
spark = (
    SparkSession.builder
    .appName("LendingRisk")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

Spark version: 4.2.0


##  Helper Functions


In [4]:
import warnings
def save_spark_df_to_csv(df, file_path):
    """
    Save a Spark DataFrame as a single CSV file.
    Suitable for small-to-medium datasets.
    """

    df.toPandas().to_csv(
            file_path,
            index=False,
            encoding="utf-8"
        )

    print(f"Saved: {file_path}")

##  Read Source Data


In [5]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(INPUT_FILE))
)

print(f"Raw record count: {raw_df.count()}")

raw_df.show(
    truncate=False
)

Raw record count: 6
+-----------+----------------+------+----------+-------------+---------------+--------------------+
|facility_id|customer_segment|metric|period_end|actual_value |threshold_value|source_updated_at   |
+-----------+----------------+------+----------+-------------+---------------+--------------------+
|F001       |SME             |PD    |2026-09-30|3.2          |3.0            |2026-10-01T05:00:00Z|
|F002       |Corporate       |PD    |2026-09-30|2.7          |3.0            |2026-10-01T05:05:00Z|
|F003       |SME             |PD    |2026-09-30|not_available|3.0            |2026-10-01T05:10:00Z|
|F004       |NULL            |PD    |2026-09-30|3.1          |3.0            |2026-10-01T05:15:00Z|
|F005       |Corporate       |PD    |2026-09-30|3.1          |3.0            |2026-10-01T05:20:00Z|
|F005       |Corporate       |PD    |2026-09-30|3.5          |3.0            |2026-10-01T06:20:00Z|
+-----------+----------------+------+----------+-------------+---------------+--

##  Execute Transformations : Transform, validate and deduplicate


In [6]:
detail_df, rejected_df, duplicates_df = transform_data(raw_df)

print(f"Valid detail rows      : {detail_df.count()}")
print(f"Rejected rows          : {rejected_df.count()}")
print(f"Superseded duplicates  : {duplicates_df.count()}")

Valid detail rows      : 3
Rejected rows          : 2
Superseded duplicates  : 1


##  Build the report summary


In [7]:
summary_df = make_summary(detail_df)

print("Detail Data")
detail_df.show(truncate=False)

print("Rejected Data")
rejected_df.select(
    "facility_id",
    "rejection_reason"
).show(truncate=False)

print("Duplicate Data")
duplicates_df.show(truncate=False)

print("Summary Data")
summary_df.show(truncate=False)

Detail Data
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+
|facility_id|customer_segment|metric|period_end|actual_value|threshold_value|source_updated_at  |is_breach|
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+
|F001       |SME             |PD    |2026-09-30|3.2         |3.0            |2026-10-01 07:00:00|true     |
|F002       |Corporate       |PD    |2026-09-30|2.7         |3.0            |2026-10-01 07:05:00|false    |
|F005       |Corporate       |PD    |2026-09-30|3.5         |3.0            |2026-10-01 08:20:00|true     |
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+

Rejected Data
+-----------+----------------------------------------------+
|facility_id|rejection_reason                              |
+-----------+----------------------------------------------+
|F003       |invalid actual_value 

## Export CSV files for Power BI
Spark writes a folder containing CSV part files for each output.

In [8]:
save_spark_df_to_csv(
    raw_df,
    OUTPUT_DIR / "raw.csv"
)

save_spark_df_to_csv(
    detail_df,
    OUTPUT_DIR / "detail.csv"
)

save_spark_df_to_csv(
    rejected_df,
    OUTPUT_DIR / "rejected.csv"
)

save_spark_df_to_csv(
    duplicates_df,
    OUTPUT_DIR / "duplicates.csv"
)

save_spark_df_to_csv(
    summary_df,
    OUTPUT_DIR / "summary.csv"
)

print(
    f"All output files successfully saved to: {OUTPUT_DIR}"
)

Saved: C:\lending_risk_overview\data\output\raw.csv
Saved: C:\lending_risk_overview\data\output\detail.csv
Saved: C:\lending_risk_overview\data\output\rejected.csv
Saved: C:\lending_risk_overview\data\output\duplicates.csv
Saved: C:\lending_risk_overview\data\output\summary.csv
All output files successfully saved to: C:\lending_risk_overview\data\output


##  UnitTest Validations
Run pytest in a terminal from the project folder: `python -m pytest -q`.